# CEG5302 Group 14: Multi-Objective Optimization

**Assignment:** Group 14 → **RCM13, Gear Box Design**, checked against the course assignment sheet (`Sheet1!A15:C15`) on 4 October 2026.

This notebook implements NSGA-II and constraint-dominance NSGA-II from first principles using NumPy. The three course-supplied problem definitions are preserved verbatim. No external images, optimizer package, data files or local Python modules are required.

**Run:** Use Python 3 with `numpy`, `matplotlib` and a Jupyter environment. Select **Restart Kernel and Run All**. All plots, tables and final design matrices are generated during execution. The PDF report summarizes this full run; do not change the experiment settings without updating the report.

**Reproducibility:** five predeclared tuning seeds (14, 114, 214, 314, 414), plus five unseen RCM holdout seeds (1014, 1114, 1214, 1314, 1414). `pc` is crossover probability per mating pair; `pm` is mutation probability per variable; `max_gen` is the number of offspring generations. The evaluation budget is `N*(G+1)`, including initialization. Wall time depends on the machine.

**RCM source convention:** the main run uses the course supplement, p. 9, literally: `14.9334/x3` in the weight proxy, `x3 in {17, 28}`, stress caps 1100/850. Public code differs. Source-specific optional alternatives are clearly separated in the model class and are not pooled into the main results. If the instructor provides a correction, change `RCM_FORMULATION` and rerun the entire analysis.

**Development assistance:** AI was used to help draft implementation and explanations. Algorithm, model and metric checks are included; all numerical results in this notebook are computed, not invented. Each group member should understand and verify the work before submission.



**Deadline**: <u>November 9, 2026 at 11:00 p.m.</u> (SGT)

## Guidelines for Submission

### A. Submission Package
1. Submit **one ZIP file per group** named **`EC_Group_<xy>.zip`**, where `<xy>` is your group number.  
   *Example:* `EC_Group_22.zip`
2. The ZIP must contain the following files:
   - **`CEG5302_Group_Project_<xy>.ipynb`** — your main Jupyter Notebook  
   - **`report_<xy>.pdf`** — your 2–3 page report  
   *Example:* `CEG5302_Group_Project_22.ipynb` and `report_22.pdf`

### B. General Requirements
1. Each group must submit **only one** ZIP.  
2. The Jupyter Notebook must clearly display **Python 3 code** and its **generated results**.
3. Your completed notebook must be **fully executable from top to bottom** and reproduce all submitted figures directly from code.  
   Do **not** replace generated result plots with static or external images. The reference figures in this template are illustrative and embedded in the notebook.  
   <span style="color:red;">Files with the `.py` extension are not accepted.</span>
4. Before submitting, **rerun the entire notebook** to confirm it executes without errors.  
   Code that cannot be executed or produces errors will result in **point deductions**.

### C. File-Naming Examples
- Group 07 → `EC_Group_07.zip` → contains `CEG5302_Group_Project_07.ipynb`, `report_07.pdf`  


*Course reference images are not imported. The figures below are generated from the executed code.*

In [ ]:
import numpy as np

In [ ]:
import os
import sys
import json
import csv
import time
from pathlib import Path
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

GROUP = 14
SEEDS = [14, 114, 214, 314, 414]
RCM_FORMULATION = "literal_pdf"
# No external files are required. During the build only, this environment
# variable exports the same generated figures/data used in the PDF report.
ARTIFACT_DIR = os.environ.get("CEG5302_ARTIFACT_DIR")
if ARTIFACT_DIR:
    ARTIFACT_DIR = Path(ARTIFACT_DIR)
    ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
np.random.seed(14)
plt.rcParams.update({"figure.dpi": 115, "savefig.dpi": 190,
                     "font.size": 10, "axes.spines.top": False,
                     "axes.spines.right": False, "axes.grid": True,
                     "grid.alpha": .18})
COLORS = ["#156082", "#E97132", "#196B24", "#A02B93", "#B8860B"]
print("Group", GROUP, "| Python", sys.version.split()[0],
      "| NumPy", np.__version__, "| Matplotlib", matplotlib.__version__)
print("Seeds:", SEEDS, "| RCM formulation:", RCM_FORMULATION)


def finish_figure(fig, name):
    fig.tight_layout()
    if ARTIFACT_DIR:
        fig.savefig(ARTIFACT_DIR / (name + ".png"), bbox_inches="tight")
    plt.show()
    plt.close(fig)


def show_table(headers, rows):
    def fmt(value):
        if isinstance(value, (float, np.floating)):
            return f"{value:.5g}"
        return str(value)
    text = "| " + " | ".join(headers) + " |\n"
    text += "| " + " | ".join(["---"] * len(headers)) + " |\n"
    text += "\n".join("| " + " | ".join(fmt(v) for v in row) + " |" for row in rows)
    display(Markdown(text))


def run_case(problem, optimizer_type, config, seed):
    arguments = dict(config)
    generations = arguments.pop("max_gen")
    optimizer = optimizer_type(**arguments, seed=seed, record_every=50)
    started = time.perf_counter()
    X, F = optimizer.run(problem, max_gen=generations)
    elapsed = time.perf_counter() - started
    assert X.shape == (arguments["pop_size"], problem.n_var)
    assert F.shape == (arguments["pop_size"], problem.n_obj)
    assert np.all(X >= problem.lower) and np.all(X <= problem.upper)
    assert np.isfinite(F).all()
    assert optimizer.n_eval == arguments["pop_size"] * (generations + 1)
    result = dict(seed=seed, config=dict(config), X=X, F=F,
                  CV=optimizer.final_cv, C=optimizer.final_constraints,
                  history=optimizer.history, seconds=elapsed,
                  n_eval=optimizer.n_eval)
    return result


def feasible_front(result):
    feasible = result["F"][result["CV"] == 0.0]
    return feasible[non_dominated(feasible)]


def reference_segments(reference):
    ordered = reference[np.argsort(reference[:, 0])]
    # A numerical plotting convention, used only to avoid joining PF gaps.
    jumps = np.linalg.norm(np.diff(ordered, axis=0), axis=1)
    boundaries = np.flatnonzero(jumps > .035) + 1
    return np.split(ordered, boundaries)


def segment_coverage(F, reference, lo, hi, tolerance=.02):
    """Descriptive diagnostic: a segment is hit within normalized distance .02.

    This is not a course marking threshold or a proof of full segment coverage.
    """
    if not len(F):
        return 0
    count = 0
    for segment in reference_segments(reference):
        A = normalize_objectives(segment, lo, hi)
        B = normalize_objectives(F, lo, hi)
        count += int(np.min(_nearest_distances(A, B)) <= tolerance)
    return count


def draw_reference(ax, reference, label="Reference front"):
    for i, piece in enumerate(reference_segments(reference)):
        ax.plot(piece[:, 0], piece[:, 1], c="#B52A35", lw=1.5,
                label=label if i == 0 else None)


def mean_sd(values):
    return float(np.mean(values)), float(np.std(values, ddof=1))


def json_ready(value):
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, dict):
        return {str(k): json_ready(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_ready(v) for v in value]
    return value


## Part 1. Continuous Multi-Objective Optimization
In this part, we have implemented the two-objective ZDT3 problem [1] and the vehicle crashworthiness design problem [2], and provided examples on how to use them. <u>**Your task is to implement NSGA-II and use it to solve these two problems**</u>.

[1] Zitzler, E., Deb, K., & Thiele, L. (2000). Comparison of multiobjective evolutionary algorithms: Empirical results. Evolutionary computation, 8(2), 173-195.

[2] Liao, X., Li, Q., Yang, X., Zhang, W., & Li, W. (2008). Multiobjective optimization for crash safety design of vehicles using a stepwise regression model. Structural and multidisciplinary optimization, 35, 561-569.

### Introduction to a Two-Objective Test Problem: ZDT3 
The ZDT test suite [1] is a well-known set of benchmark problems in multi-objective optimization. It was introduced by Zitzler, Deb, and Thiele in 2000 to evaluate the performance of multi-objective optimization algorithms. ZDT3 is one of the test problems in the ZDT test suite.

- Number of objectives: 2
- Number of decision variables: 30

$$
\begin{aligned}
&\mathrm{minimize} \left\{
\begin{array}{l}
f_1(\mathbf{x})=x_1 \\
f_2(\mathbf{x})=g(\mathbf{x}) h(\mathbf{x})
\end{array}\:,\right. \\
&\mathrm{where} \enspace g(\mathbf{x})=1+\frac{9}{n-1} \sum_{i=2}^n x_i, \\
& \qquad \enspace h(\mathbf{x})=1-\sqrt{f_1(\mathbf{x}) / g(\mathbf{x})}-\left(f_1(\mathbf{x}) / g(\mathbf{x})\right) \sin \left(10 \pi f_1(\mathbf{x})\right),\\
& \qquad \enspace 0 \leq x_i \leq 1, \enspace i=1,\ldots,n.
\end{aligned}
$$

[1] E. Zitzler, K. Deb, and L. Thiele, Comparison of multiobjective evolutionary algorithms: Empirical results, Evolutionary computation, 2000, 8(2): 173-195.

In [ ]:
class ZDT3():

    def __init__(self):
        self.name = 'ZDT3'
        self.n_obj = 2  # number of objectives
        self.n_var = 30  # number of decision variables

        self.lower = np.zeros(self.n_var)  # lower bound of decision variables
        self.upper = np.ones(self.n_var)  # upper bound of decision variables

    def evaluate(self, x):
        pop_size = len(x)
        f = np.zeros((pop_size, self.n_obj)) # for each chromosom, we have 2 objective function evals.
        f[:, 0] = x[:, 0]
        g = 1.0 + 9.0 * np.sum(x[:, 1:], axis=1) / (self.n_var - 1)
        h = 1 - np.power(
            f[:, 0] * 1.0 / g,
            0.5) - (f[:, 0] * 1.0 / g) * np.sin(10 * np.pi * f[:, 0])
        f[:, 1] = g * h

        return f

In [ ]:
# Create an instance of ZDT3
prob = ZDT3()
print(f"Problem = {prob.name}")
print(f"Number of objectives = {prob.n_obj}")
print(f"Number of variables = {prob.n_var}")
print(f"Lower bounds = {prob.lower}")
print(f"Upper bounds = {prob.upper}")

In [ ]:
# Initialize a population with 1 solution and evaluate it
pop_size = 1
x = prob.lower + (prob.upper - prob.lower) * np.random.rand(
    pop_size, prob.n_var)

print(f"x: {x}")

f = prob.evaluate(x)
print(f"f(x) = {f}")


In [ ]:
# Initialize a population with 5 solutions and evaluate them
pop_size = 5
x = prob.lower + (prob.upper - prob.lower) * np.random.rand(
    pop_size, prob.n_var)


f = prob.evaluate(x)
print(f"f(x) = {f}")

### Task: Implement NSGA-II and Solve ZDT3 (30 marks)

Your task is **to implement NSGA-II, use it to solve ZDT3, obtain the final population, and visualize the final population.** 

To help you get started, we have provided skeleton code for NSGA-II, which serves as a starting point for you to add your own implementation. You may edit the **TODO sections and method signatures**, and add helper methods inside the `NSGA2` or `CNSGA2` classes when needed. Do not modify the supplied problem definitions (`ZDT3`, `CrashworthinessDesign`, and `MW7`).  Additionally, we have also provided you with code to create instances of the NSGA2() and ZDT3() classes.

An example of the expected output is shown in the following figure, which displays the true Pareto front of ZDT3 (represented by red lines) and the final population obtained by NSGA-II (represented by grey circles). Your final solutions should be similar to these grey circles.



It is crucial to ensure that your code runs successfully, as failing to do so may result in a zero mark for this task. If you face difficulties in implementing NSGA-II in this part, you can refer to available source codes online, such as [PlatEMO](https://github.com/BIMK/PlatEMO/blob/master/PlatEMO/Algorithms/Multi-objective%20optimization/NSGA-II/NSGAII.m) and [pymoo](https://github.com/anyoptimization/pymoo/blob/main/pymoo/algorithms/moo/nsga2.py), or seek assistance from ChatGPT.

*Course reference images are not imported. The figures below are generated from the executed code.*

### Implementation and independent checks

All objectives are minimized. Rank zero contains solutions that are not dominated by another solution in the current population. Crowding distance uses each objective's range within its own front; constant objectives contribute zero. Parent and offspring populations are merged, and complete fronts are retained before crowding-distance truncation of the last front. Random ties are reproducible.

SBX and polynomial mutation enforce variable bounds. The optional problem `repair` method enforces a discrete domain before evaluation. The algorithm is not specialized to ZDT3 and supports two or three objectives without changing its selection logic.


In [ ]:
"""Pure NumPy NSGA-II and constraint-dominance NSGA-II.

The problem interface is n_var, n_obj, lower, upper, evaluate(X), with
optional repair(X). evaluate returns F or (F, C); C contains nonnegative
constraint violation magnitudes, never raw signed constraint functions.
"""

import numpy as np


class NSGA2:
    """Elitist NSGA-II for minimization, with bounded real-coded variation.

    pc is the probability of SBX for a mating pair. pm is the independent
    mutation probability per variable (None means 1 / problem.n_var).
    max_gen counts offspring generations, so evaluations are N * (G + 1).
    The base class uses objective dominance; CNSGA2 also uses constraints.
    Every run resets the generator to seed, enabling reproducible reruns.
    """

    def __init__(self, pop_size=100, pc=0.9, pm=None, eta_c=20,
                 eta_m=20, seed=0, record_every=50):
        if not isinstance(pop_size, (int, np.integer)) or pop_size < 2:
            raise ValueError("pop_size must be an integer of at least 2")
        if not 0 <= pc <= 1 or (pm is not None and not 0 <= pm <= 1):
            raise ValueError("pc and pm must be probabilities in [0, 1]")
        if eta_c < 0 or eta_m < 0:
            raise ValueError("distribution indices must be nonnegative")
        if not isinstance(record_every, (int, np.integer)) or record_every < 1:
            raise ValueError("record_every must be a positive integer")
        self.pop_size = int(pop_size)
        self.pc = float(pc)
        self.pm = None if pm is None else float(pm)
        self.eta_c = float(eta_c)
        self.eta_m = float(eta_m)
        self.seed = seed
        self.record_every = int(record_every)
        self.rng = np.random.default_rng(seed)
        self.n_eval = 0
        self.history = []
        self.final_cv = None
        self.final_constraints = None

    @staticmethod
    def _objective_dominance(F):
        """D[i,j] means i dominates j; equal objective vectors do not dominate."""
        F = np.asarray(F, dtype=float)
        n = len(F)
        no_worse = np.ones((n, n), dtype=bool)
        strictly_better = np.zeros((n, n), dtype=bool)
        for column in F.T:
            no_worse &= column[:, None] <= column[None, :]
            strictly_better |= column[:, None] < column[None, :]
        return no_worse & strictly_better

    def _dominance_matrix(self, F, CV=None):
        return self._objective_dominance(F)

    def non_dominated_sort(self, F, CV=None):
        """Return zero-based ranks and a list of fronts of row indices."""
        F = np.asarray(F, dtype=float)
        if F.ndim != 2 or not np.isfinite(F).all():
            raise ValueError("F must be a finite two-dimensional array")
        dominates = self._dominance_matrix(F, CV)
        domination_count = dominates.sum(axis=0)
        rank = np.full(len(F), -1, dtype=int)
        front = np.flatnonzero(domination_count == 0)
        fronts = []
        while front.size:
            rank[front] = len(fronts)
            fronts.append(front)
            domination_count -= dominates[front].sum(axis=0)
            front = np.flatnonzero((domination_count == 0) & (rank < 0))
        if np.any(rank < 0):
            raise RuntimeError("Dominance sorting did not assign every row")
        return rank, fronts

    @staticmethod
    def crowding_distance(F, fronts):
        """Standard front-wise, objective-range-normalized crowding distance.

        An objective constant across a front contributes zero; it cannot
        distinguish any member and should not create arbitrary boundaries.
        """
        F = np.asarray(F, dtype=float)
        distance = np.zeros(len(F), dtype=float)
        for front in fronts:
            front = np.asarray(front, dtype=int)
            if len(front) <= 2:
                distance[front] = np.inf
                continue
            for j in range(F.shape[1]):
                ordered = front[np.argsort(F[front, j], kind="mergesort")]
                span = F[ordered[-1], j] - F[ordered[0], j]
                if span <= 0:
                    continue
                distance[ordered[[0, -1]]] = np.inf
                distance[ordered[1:-1]] += (
                    F[ordered[2:], j] - F[ordered[:-2], j]
                ) / span
        return distance

    def fitness_assignment(self, F, CV=None):
        ranks, fronts = self.non_dominated_sort(F, CV)
        return ranks, self.crowding_distance(F, fronts)

    def tournament_selection(self, population_x, ranks, crowding):
        """Binary tournaments: lower rank, then larger crowding, random ties."""
        n = len(population_x)
        contestants = self.rng.integers(n, size=(self.pop_size, 2))
        a, b = contestants.T
        same_rank = ranks[a] == ranks[b]
        same_distance = crowding[a] == crowding[b]
        a_wins = (ranks[a] < ranks[b]) | (
            same_rank & ((crowding[a] > crowding[b]) |
                         (same_distance & (self.rng.random(len(a)) < 0.5)))
        )
        return np.asarray(population_x)[np.where(a_wins, a, b)].copy()

    def _repair(self, X, prob):
        original_shape = np.asarray(X).shape
        X = np.clip(X, prob.lower, prob.upper)
        if hasattr(prob, "repair"):
            repaired = prob.repair(X.copy())
            X = np.asarray(repaired, dtype=float)
        if X.ndim != 2 or X.shape != original_shape or X.shape[1] != prob.n_var:
            raise ValueError("repair must preserve the (population, n_var) shape")
        if (not np.isfinite(X).all() or np.any(X < prob.lower)
                or np.any(X > prob.upper)):
            raise ValueError("repair produced nonfinite or out-of-bounds variables")
        return X

    def initialize(self, prob):
        X = prob.lower + (prob.upper - prob.lower) * self.rng.random(
            (self.pop_size, prob.n_var))
        return self._repair(X, prob)

    def crossover(self, parents, prob):
        """Deb's bounded simulated binary crossover, including odd sizes."""
        parents = np.asarray(parents, dtype=float)
        original_size = len(parents)
        if original_size % 2:
            parents = np.vstack((parents, parents[:1]))
        p1, p2 = parents[0::2], parents[1::2]
        q1, q2 = p1.copy(), p2.copy()
        y1, y2 = np.minimum(p1, p2), np.maximum(p1, p2)
        delta = y2 - y1
        active = ((self.rng.random((len(p1), 1)) < self.pc)
                  & (self.rng.random(p1.shape) < 0.5)
                  & (delta > 1e-14)
                  & (np.asarray(prob.upper) > np.asarray(prob.lower)))
        # Inactive coordinates use a safe denominator and are never replaced.
        denominator = np.where(active, delta, 1.0)
        random_value = self.rng.random(p1.shape)
        power = 1.0 / (self.eta_c + 1.0)

        beta_lower = 1.0 + 2.0 * (y1 - prob.lower) / denominator
        alpha_lower = 2.0 - beta_lower ** (-(self.eta_c + 1.0))
        beta_q_lower = np.where(
            random_value <= 1.0 / alpha_lower,
            (random_value * alpha_lower) ** power,
            (1.0 / (2.0 - random_value * alpha_lower)) ** power)
        beta_upper = 1.0 + 2.0 * (prob.upper - y2) / denominator
        alpha_upper = 2.0 - beta_upper ** (-(self.eta_c + 1.0))
        beta_q_upper = np.where(
            random_value <= 1.0 / alpha_upper,
            (random_value * alpha_upper) ** power,
            (1.0 / (2.0 - random_value * alpha_upper)) ** power)
        c1 = 0.5 * (y1 + y2 - beta_q_lower * delta)
        c2 = 0.5 * (y1 + y2 + beta_q_upper * delta)
        swap = self.rng.random(p1.shape) < 0.5
        q1[active] = np.where(swap, c2, c1)[active]
        q2[active] = np.where(swap, c1, c2)[active]
        offspring = np.empty_like(parents)
        offspring[0::2], offspring[1::2] = q1, q2
        return np.clip(offspring[:original_size], prob.lower, prob.upper)

    def mutation(self, offspring, prob):
        """Bounded polynomial mutation; pm is a per-variable probability."""
        X = np.array(offspring, dtype=float, copy=True)
        probability = 1.0 / prob.n_var if self.pm is None else self.pm
        width = np.asarray(prob.upper) - np.asarray(prob.lower)
        active = (self.rng.random(X.shape) < probability) & (width > 0)
        safe_width = np.where(width > 0, width, 1.0)
        delta1 = (X - prob.lower) / safe_width
        delta2 = (prob.upper - X) / safe_width
        random_value = self.rng.random(X.shape)
        power = 1.0 / (self.eta_m + 1.0)
        lower_value = (2.0 * random_value + (1.0 - 2.0 * random_value)
                       * (1.0 - delta1) ** (self.eta_m + 1.0))
        upper_value = (2.0 * (1.0 - random_value)
                       + 2.0 * (random_value - 0.5)
                       * (1.0 - delta2) ** (self.eta_m + 1.0))
        # Each branch is meaningful only on its own half of [0,1].
        delta_q = np.where(random_value <= 0.5,
                           np.maximum(lower_value, 0.0) ** power - 1.0,
                           1.0 - np.maximum(upper_value, 0.0) ** power)
        X += np.where(active, delta_q * safe_width, 0.0)
        return self._repair(X, prob)

    def environmental_selection(self, X, F, C=None):
        """Select N elites from the combined parents and offspring."""
        X, F = np.asarray(X), np.asarray(F)
        C = (np.zeros((len(X), 0)) if C is None
             else np.asarray(C, dtype=float))
        if C.ndim == 1:
            C = C[:, None]
        CV = C.sum(axis=1)
        ranks, fronts = self.non_dominated_sort(F, CV)
        distance = self.crowding_distance(F, fronts)
        selected = []
        for front in fronts:
            remaining = self.pop_size - len(selected)
            if len(front) <= remaining:
                selected.extend(front.tolist())
            else:
                # Random ordering breaks equal-distance ties reproducibly.
                shuffled = self.rng.permutation(front)
                best = shuffled[np.argsort(-distance[shuffled], kind="mergesort")]
                selected.extend(best[:remaining].tolist())
                break
            if len(selected) == self.pop_size:
                break
        selected = np.asarray(selected, dtype=int)
        return X[selected].copy(), F[selected].copy(), C[selected].copy()

    def _evaluate(self, prob, X):
        result = prob.evaluate(X)
        if isinstance(result, tuple):
            if len(result) != 2:
                raise ValueError("evaluate must return F or (F, C)")
            F, C = result
            C = np.asarray(C, dtype=float)
            if C.ndim == 1:
                C = C[:, None]
        else:
            F = result
            C = np.zeros((len(X), 0))
        F = np.asarray(F, dtype=float)
        if F.shape != (len(X), prob.n_obj) or not np.isfinite(F).all():
            raise ValueError("evaluate returned invalid objective values or shape")
        if (C.ndim != 2 or C.shape[0] != len(X)
                or not np.isfinite(C).all() or np.any(C < 0)):
            raise ValueError("C must contain finite nonnegative violation magnitudes")
        self.n_eval += len(X)
        return F, C

    def _record(self, generation, F, C):
        CV = C.sum(axis=1)
        self.history.append({
            "gen": int(generation), "n_eval": int(self.n_eval),
            "feasible_ratio": float(np.mean(CV == 0.0)),
            "cv_min": float(np.min(CV)),
            "F": F.copy(), "CV": CV.copy(),
        })

    def run(self, prob, max_gen=500):
        if not isinstance(max_gen, (int, np.integer)) or max_gen < 0:
            raise ValueError("max_gen must be a nonnegative integer")
        lower, upper = np.asarray(prob.lower), np.asarray(prob.upper)
        if (lower.shape != (prob.n_var,) or upper.shape != (prob.n_var,)
                or not np.isfinite(lower).all() or not np.isfinite(upper).all()
                or np.any(lower > upper)):
            raise ValueError("problem bounds must be finite valid n_var-vectors")
        self.rng = np.random.default_rng(self.seed)
        self.n_eval, self.history = 0, []
        population_x = self.initialize(prob)
        population_fx, population_c = self._evaluate(prob, population_x)
        self._record(0, population_fx, population_c)
        for generation in range(1, max_gen + 1):
            ranks, distance = self.fitness_assignment(
                population_fx, population_c.sum(axis=1))
            parents = self.tournament_selection(population_x, ranks, distance)
            offspring_x = self.mutation(self.crossover(parents, prob), prob)
            offspring_fx, offspring_c = self._evaluate(prob, offspring_x)
            population_x, population_fx, population_c = self.environmental_selection(
                np.vstack((population_x, offspring_x)),
                np.vstack((population_fx, offspring_fx)),
                np.vstack((population_c, offspring_c)))
            if generation % self.record_every == 0 or generation == max_gen:
                self._record(generation, population_fx, population_c)
        self.final_cv = population_c.sum(axis=1).copy()
        self.final_constraints = population_c.copy()
        return population_x, population_fx




In [ ]:
"""Reproducible, NumPy-only minimization metrics for the course experiments.

Use ONE frozen set of objective bounds (lo, hi), reference points and, for HV,
reference vector for every configuration being compared. Do not normalize each
run separately. Bounds are scaling constants, not a request to clip observations.
Only feasible observations should be passed for constrained problems.
"""

import numpy as np


def _matrix(values, n_obj=None, name="points"):
    values = np.asarray(values, dtype=float)
    if values.size == 0 and values.ndim == 1:
        values = values.reshape(0, 0 if n_obj is None else n_obj)
    if values.ndim != 2:
        raise ValueError(f"{name} must be a 2D array")
    if n_obj is not None and values.shape[1] != n_obj:
        raise ValueError(f"{name} must have {n_obj} objectives")
    if not np.all(np.isfinite(values)):
        raise ValueError(f"{name} contains non-finite values")
    return values


def non_dominated(F):
    """Boolean mask of Pareto-nondominated rows; minimize every objective.

    Equal duplicates do not dominate one another and receive the same mask.
    The 2-objective implementation sorts in O(N log N), permitting dense MW7
    reference construction; other dimensions use an O(N**2) comparison loop.
    """
    F = _matrix(F)
    n, m = F.shape
    if n == 0:
        return np.zeros(0, dtype=bool)
    if m == 0:
        raise ValueError("At least one objective is required")
    if m == 1:
        return F[:, 0] == np.min(F[:, 0])
    if m == 2:
        unique, inverse = np.unique(F, axis=0, return_inverse=True)
        previous_best = np.r_[np.inf, np.minimum.accumulate(unique[:-1, 1])]
        return (unique[:, 1] < previous_best)[inverse]
    keep = np.ones(n, dtype=bool)
    for i in range(n):
        keep[i] = not np.any(np.all(F <= F[i], axis=1)
                             & np.any(F < F[i], axis=1))
    return keep


def normalize_objectives(F, lo, hi):
    """Apply fixed affine objective scaling; never silently clip values."""
    lo, hi = np.asarray(lo, dtype=float), np.asarray(hi, dtype=float)
    if lo.ndim != 1 or hi.shape != lo.shape or lo.size == 0:
        raise ValueError("lo and hi must be matching nonempty vectors")
    if not np.all(np.isfinite(lo)) or not np.all(np.isfinite(hi)):
        raise ValueError("Bounds must be finite")
    if np.any(hi <= lo):
        raise ValueError("Every hi must be strictly greater than lo")
    return (_matrix(F, lo.size) - lo) / (hi - lo)


def _nearest_distances(A, B, chunk_size=256):
    if len(B) == 0:
        return np.full(len(A), np.inf)
    result = np.empty(len(A))
    for start in range(0, len(A), chunk_size):
        delta = A[start:start + chunk_size, None, :] - B[None, :, :]
        result[start:start + chunk_size] = np.sqrt(
            np.min(np.sum(delta * delta, axis=2), axis=1))
    return result


def normalized_igd(ref, F, lo, hi):
    """Mean normalized Euclidean distance from reference front to approximation.

    Lower is better relative to the SAME reference sample and normalization.
    This is mean distance (IGD with p=1), not RMS distance. Empty F returns inf;
    an empty reference raises ValueError rather than inventing an ideal front.
    """
    R = normalize_objectives(ref, lo, hi)
    A = normalize_objectives(F, lo, hi)
    if not len(R):
        raise ValueError("IGD requires a nonempty reference front")
    return float(np.mean(_nearest_distances(R, A)))


def normalized_gd(ref, F, lo, hi):
    """Mean normalized Euclidean distance from approximation to reference.

    This is mean distance (GD with p=1). It measures proximity only: one good
    point can have excellent GD while covering almost none of the front.
    Empty F returns inf; an empty reference raises ValueError.
    """
    R = normalize_objectives(ref, lo, hi)
    A = normalize_objectives(F, lo, hi)
    if not len(R):
        raise ValueError("GD requires a nonempty reference front")
    if not len(A):
        return float("inf")
    return float(np.mean(_nearest_distances(A, R)))


def normalized_spacing(F, lo, hi):
    """Return mean_nn, cv_nn and n using normalized Euclidean NN distances.

    cv_nn is population standard deviation divided by mean NN distance.
    Neither statistic alone establishes better diversity: clustering may give
    a tiny CV, while an isolated outlier may raise mean_nn. Read them together
    with coverage, feasible nondominated count and HV/IGD. Duplicates remain
    visible as zero NN distances. For <2 points both statistics are NaN; if all
    NN distances are zero, mean_nn=0 and cv_nn=NaN (division is undefined).
    """
    A = normalize_objectives(F, lo, hi)
    n = len(A)
    if n < 2:
        return {"mean_nn": float("nan"), "cv_nn": float("nan"), "n": n}
    d = np.sqrt(np.sum((A[:, None, :] - A[None, :, :]) ** 2, axis=2))
    np.fill_diagonal(d, np.inf)
    nearest = np.min(d, axis=1)
    mean = float(np.mean(nearest))
    return {"mean_nn": mean,
            "cv_nn": float(np.std(nearest) / mean) if mean > 0 else float("nan"),
            "n": n}


def _hypervolume_2d(F, reference):
    """Area of the union of minimization rectangles, with valid corners only."""
    if len(F) == 0:
        return 0.0
    order = np.argsort(F[:, 0], kind="stable")
    sorted_F = F[order]
    best_y = np.minimum.accumulate(sorted_F[:, 1])
    widths = np.diff(np.r_[sorted_F[:, 0], reference[0]])
    return float(np.dot(widths, reference[1] - best_y))


def hypervolume_3d(F, reference=1.1):
    """EXACT 3D dominated hypervolume by an x sweep and exact 2D slices.

    F must contain normalized, feasible objective vectors, all minimized.
    Feasibility cannot be inferred here: the caller must select feasible rows.
    The function defensively removes dominated rows and equal duplicates.
    reference is a fixed scalar or length-3 vector, normally [1.1]*3 shared by
    ALL configurations. Points at or beyond reference in ANY coordinate cannot
    dominate a positive-volume box and are excluded, not projected inwards.
    Negative coordinates are allowed and not clipped; they mean an observation
    improved on the fixed lower normalization bound. The returned volume is in
    normalized objective units and is NOT divided by 1.1**3. This is not a Monte
    Carlo estimate. Larger is better with a common scale and reference point.
    """
    F = _matrix(F, 3)
    reference = np.asarray(reference, dtype=float)
    if reference.ndim == 0:
        reference = np.full(3, float(reference))
    if reference.shape != (3,) or not np.all(np.isfinite(reference)):
        raise ValueError("reference must be a finite scalar or length-3 vector")
    F = F[np.all(F < reference, axis=1)]
    if not len(F):
        return 0.0
    F = np.unique(F[non_dominated(F)], axis=0)
    x_values = np.unique(F[:, 0])
    widths = np.diff(np.r_[x_values, reference[0]])
    volume = 0.0
    for x, width in zip(x_values, widths):
        volume += width * _hypervolume_2d(F[F[:, 0] <= x, 1:], reference[1:])
    return float(volume)


def normalized_hypervolume_3d(F, lo, hi, reference=1.1):
    """Convenience wrapper retaining a fixed shared normalization and reference."""
    return hypervolume_3d(normalize_objectives(F, lo, hi), reference)


ZDT3_INTERVALS = np.array([
    [0.0, 0.0830015349],
    [0.1822287280, 0.2577623634],
    [0.4093136748, 0.4538821041],
    [0.6183967944, 0.6525117038],
    [0.8233317983, 0.8518328654],
])


def zdt3_reference_front(n_points=2000):
    """Sample the five analytic ZDT3 front segments (g=1).

    Endpoints are standard roots rounded to 10 decimals. Samples are allocated
    approximately uniformly by objective-space arc length, avoiding excessive
    weight near steep/short segments. Arc length uses a dense deterministic
    numerical table; front coordinates themselves use the exact g=1 formula.
    Segment starts at equal-f2 joins are interpreted as the front's closure.
    """
    if int(n_points) != n_points or n_points < 10:
        raise ValueError("n_points must be an integer >=10")
    n_points = int(n_points)
    tables, lengths = [], []
    for left, right in ZDT3_INTERVALS:
        t = np.linspace(0.0, 1.0, 4001)
        x = left + (right - left) * (t ** 2 if left == 0 else t)
        y = 1.0 - np.sqrt(x) - x * np.sin(10.0 * np.pi * x)
        distance = np.r_[0.0, np.cumsum(np.hypot(np.diff(x), np.diff(y))) ]
        tables.append((x, distance))
        lengths.append(distance[-1])
    allocation = (n_points - 10) * np.array(lengths) / np.sum(lengths)
    counts = np.floor(allocation).astype(int) + 2
    remainder = n_points - counts.sum()
    if remainder:
        counts[np.argsort(-(allocation - np.floor(allocation)))[:remainder]] += 1
    fronts = []
    for count, (x, distance) in zip(counts, tables):
        sampled_x = np.interp(np.linspace(0, distance[-1], count), distance, x)
        sampled_y = 1.0 - np.sqrt(sampled_x) - sampled_x * np.sin(10*np.pi*sampled_x)
        fronts.append(np.column_stack((sampled_x, sampled_y)))
    return np.vstack(fronts)


def mw7_reference_front(n_angles=100001, n_points=None):
    """Dense NUMERICAL reference front derived from the supplied MW7 equations.

    Set theta=atan2(f2,f1) in [0,pi/2]. The radial objectives have radius g3>=1.
    Feasibility requires 1.15-0.2*sin(4*theta)**8 <= g3 <=
    1.2+0.4*sin(4*theta)**16. Thus the feasible radial lower boundary has
    r=max(1, 1.15-0.2*sin(4*theta)**8). Some points on that boundary are globally
    dominated, so they MUST be filtered. We sample the boundary and retain its
    globally nondominated subset. This is a dense numerical approximation to
    the constrained front, not a claim that every radial-boundary point is PF.

    The bound g3=1 is attainable by x[i]=1-(x[i-1]-.5)**2. Required radii up to
    1.15 are attainable by lowering only the last variable by sqrt((r-1)/2),
    which remains inside [0,1]. Optional n_points uniformly subsamples the
    retained index sequence (not a curve interpolated across front gaps).
    """
    if int(n_angles) != n_angles or n_angles < 1001:
        raise ValueError("n_angles must be an integer >=1001")
    theta = np.linspace(0.0, np.pi / 2, int(n_angles))
    wave = np.sin(4 * theta) ** 8
    radius = np.maximum(1.0, 1.15 - 0.2 * wave)
    F = np.column_stack((radius * np.cos(theta), radius * np.sin(theta)))
    # Exact endpoints avoid floating residuals from cos(pi/2).
    F[0, 1] = 0.0
    F[-1, 0] = 0.0
    F = F[non_dominated(F)]
    F = F[np.argsort(F[:, 0])]
    if n_points is not None:
        if int(n_points) != n_points or n_points < 2:
            raise ValueError("n_points must be an integer >=2")
        if n_points < len(F):
            F = F[np.round(np.linspace(0, len(F)-1, int(n_points))).astype(int)]
    return F


In [ ]:
# Small independent examples check logic, not merely successful execution.
check = NSGA2(pop_size=4, seed=14)
example_F = np.array([[1.,8.], [2.,5.], [3.,9.], [4.,3.]])
rank, fronts = check.non_dominated_sort(example_F)
assert rank.tolist() == [0,0,1,0]
assert non_dominated(example_F).tolist() == [True,True,False,True]
assert np.isclose(hypervolume_3d([[.2,.3,.4]], reference=1.), .8*.7*.6)
assert np.isclose(hypervolume_3d([[.2,.2,.8], [.8,.2,.2]], reference=1.), .224)
points = np.array([[0.,1.], [.5,.5], [1.,0.]])
assert np.isclose(normalized_igd(points, points, [0,0], [1,1]), 0)
small = NSGA2(pop_size=9, seed=14)
x1, f1 = small.run(ZDT3(), max_gen=3)
x2, f2 = small.run(ZDT3(), max_gen=3)
assert np.array_equal(x1,x2) and np.array_equal(f1,f2)
assert small.n_eval == 36 and np.all((x1>=0)&(x1<=1))
print('Dominance, exact HV, metric identity, odd population, bounds and reproducibility checks passed.')


### Task 1 experiment: ZDT3

Baseline: N=100, G=500, pc=0.9, pm=1/30, distribution indices eta_c=eta_m=20. The reference consists of 2,000 approximately arc-length-spaced samples on the five analytic front segments (g=1). GD and IGD use mean Euclidean distance after one fixed normalization by that reference's objective ranges. Lower is better. Segment hits use a diagnostic distance of 0.02 on that scale; this is not an official grading threshold.


In [ ]:
zdt_problem = ZDT3()
zdt_config = dict(pop_size=100, pc=.9, pm=1/30, max_gen=500)
zdt_reference = zdt3_reference_front(2000)
zdt_lo, zdt_hi = zdt_reference.min(axis=0), zdt_reference.max(axis=0)
zdt_runs = [run_case(zdt_problem, NSGA2, zdt_config, seed) for seed in SEEDS]
zdt_stats = []
for r in zdt_runs:
    F = feasible_front(r)
    zdt_stats.append(dict(seed=r["seed"],
        igd=normalized_igd(zdt_reference, F, zdt_lo, zdt_hi),
        gd=normalized_gd(zdt_reference, F, zdt_lo, zdt_hi),
        segments=segment_coverage(F, zdt_reference, zdt_lo, zdt_hi),
        median_g=float(np.median(1+9*np.mean(r["X"][:, 1:], axis=1))),
        seconds=r["seconds"], n_eval=r["n_eval"]))
show_table(["Seed", "IGD (lower)", "GD (lower)", "Segments hit / 5", "Median g", "Seconds"],
           [[s[k] for k in ("seed", "igd", "gd", "segments", "median_g", "seconds")]
            for s in zdt_stats])
optimum_x, optimum_fx = zdt_runs[0]["X"], zdt_runs[0]["F"]
fig, axes = plt.subplots(1, 2, figsize=(11, 4.1))
draw_reference(axes[0], zdt_reference, "Analytic ZDT3 front")
axes[0].scatter(optimum_fx[:, 0], optimum_fx[:, 1], s=17, color=COLORS[0],
                label="Final population, seed 14")
axes[0].set(xlabel="$f_1$", ylabel="$f_2$", title="ZDT3: final population")
axes[0].legend(fontsize=8)
zdt_curve = np.array([[normalized_igd(zdt_reference, h["F"], zdt_lo, zdt_hi)
                        for h in r["history"]] for r in zdt_runs])
gens = [h["gen"] for h in zdt_runs[0]["history"]]
for i, curve in enumerate(zdt_curve):
    axes[1].plot(gens, curve, color=COLORS[i], alpha=.65, lw=1, label=str(SEEDS[i]))
axes[1].set(xlabel="Offspring generations", ylabel="Normalized IGD (log scale)",
            yscale="log", title="Convergence across five fixed seeds")
axes[1].legend(title="Seed", fontsize=8, ncol=2)
finish_figure(fig, "zdt3_results")
zmean, zsd = mean_sd([s["igd"] for s in zdt_stats])
display(Markdown(f"**Observed result.** Mean normalized IGD = **{zmean:.5f} ± {zsd:.5f}** "
    f"(sample SD, five seeds). Segment-hit counts are {[s['segments'] for s in zdt_stats]}. "
    "The small median g values near one provide a decision-space sanity check. "
    "Coverage uses a stated distance tolerance and is not a claim of exact optimality."))


### Task: Use NSGA-II to Solve a Real-World Problem: Crashworthiness Design of Vehicles (10 marks)

The automotive industry places great importance on structural optimization for crashworthiness. In a study by Liao et al. [2], the crashworthiness design of vehicles was modeled as a multi-objective optimization problem. The problem aims to minimize three objectives: weight ($f_1$), acceleration characteristics ($f_2$), and toe-board intrusion ($f_3$) of the vehicle design. The decision variables for the problem specify the thickness of five reinforced members around the frontal structure of the vehicle, respectively. You are required to use NSGA-II to optimize the following crashworthiness design problem.

- Number of objectives: 3
- Number of decision variables: 5

$$
\begin{aligned}
&\mathrm{minimize} \left\{\begin{aligned}
f_1(\mathbf{x})= \,& 1640.2823+2.3573285 x_1+2.3220035 x_2 \\
& +4.5688768 x_3+7.7213633 x_4 +4.4559504 x_5\\
f_2(\mathbf{x})= \,& 6.5856+1.15 x_1-1.0427 x_2+0.9738 x_3 \\
& +0.8364 x_4-0.3695 x_1 x_4+0.0861 x_1 x_5 \\
& +0.3628 x_2 x_4-0.1106 x_1^2-0.3437 x_3^2 \\
& +0.1764 x_4^2\\
f_3(\mathbf{x})= & -0.0551+0.0181 x_1+0.1024 x_2 \\
& +0.0421 x_3-0.0073 x_1 x_2+0.024 x_2 x_3 \\
& -0.0118 x_2 x_4-0.0204 x_3 x_4-0.008 x_3 x_5 \\
& -0.0241 x_2^2+0.0109 x_4^2\\
\end{aligned}\right., \\
& \mathrm{where} \enspace  1 \leq x_i \leq 3, \enspace i=1,\ldots,5.
\end{aligned}
$$


[2] Liao, X., Li, Q., Yang, X., Zhang, W., & Li, W. (2008). Multiobjective optimization for crash safety design of vehicles using stepwise regression model. Structural and multidisciplinary optimization, 35, 561-569.

Your task is to **instantiate the NSGA2() and CrashworthinessDesign() classes, use NSGA-II to solve the problem, obtain the final population, and visualize the final population**. 

You can refer to the example figures below for reference. The left figure displays the Pareto front approximated by combining solutions obtained by several state-of-the-art algorithms, and the right figure displays the final population obtained by NSGA-II. Your expected solutions should be similar to the right figure.


    
    


Note that while NSGA-II provides solutions that are in close proximity to the Pareto front, indicating good convergence, these solutions are not evenly distributed across the Pareto front. **What are the reasons behind the poor distribution of solutions obtained by NSGA-II? Do you have any suggestions for improving NSGA-II performance on this problem? Please share your thoughts and suggestions.**


*Course reference images are not imported. The figures below are generated from the executed code.*

In [ ]:
class CrashworthinessDesign():

    def __init__(self):
        self.name = 'Crashworthiness design of vehicles'
        self.n_obj = 3
        self.n_var = 5

        self.lower = np.full(self.n_var, 1.0)
        self.upper = np.full(self.n_var, 3.0)

    def evaluate(self, x):
        pop_size = len(x)
        f = np.zeros((pop_size, self.n_obj))

        x1 = x[:, 0]
        x2 = x[:, 1]
        x3 = x[:, 2]
        x4 = x[:, 3]
        x5 = x[:, 4]

        f[:, 0] = 1640.2823 + (2.3573285 * x1) + (2.3220035 * x2) + (
            4.5688768 * x3) + (7.7213633 * x4) + (4.4559504 * x5)
        f[:, 1] = 6.5856 + (1.15 * x1) - (1.0427 * x2) + (0.9738 * x3) + (
            0.8364 * x4) - (0.3695 * x1 * x4) + (0.0861 * x1 * x5) + (
                0.3628 * x2 * x4) - (0.1106 * x1 * x1) - (0.3437 * x3 * x3) + (
                    0.1764 * x4 * x4)
        f[:, 2] = -0.0551 + (0.0181 * x1) + (0.1024 * x2) + (0.0421 * x3) - (
            0.0073 * x1 * x2) + (0.024 * x2 * x3) - (0.0118 * x2 * x4) - (
                0.0204 * x3 * x4) - (0.008 * x3 * x5) - (0.0241 * x2 * x2) + (
                    0.0109 * x4 * x4)

        return f

In [ ]:
# Create an instance of the crashworthiness design problem
prob = CrashworthinessDesign()
print(f"Problem = {prob.name}")
print(f"Number of objectives = {prob.n_obj}")
print(f"Number of variables = {prob.n_var}")
print(f"Lower bounds = {prob.lower}")
print(f"Upper bounds = {prob.upper}")

In [ ]:
# Initialize a population with 5 solutions and evaluate them
pop_size = 5
x = prob.lower + (prob.upper - prob.lower) * np.random.rand(
    pop_size, prob.n_var)
print("x = ", x)

f = prob.evaluate(x)
print(f"f(x) = {f}")

### Task 2 experiment: crashworthiness

The same NSGA-II implementation uses N=100, G=500, pc=0.9, pm=1/5. Three objectives require three-dimensional visualization. All five runs share the pooled final-population min/max scale and HV reference (1.1,1.1,1.1). This is an empirical within-experiment comparison, not a distance to a known exact front. The coefficient of variation (CV) of normalized nearest-neighbour distances is descriptive and cannot by itself establish good coverage.


In [ ]:
crash_problem = CrashworthinessDesign()
crash_config = dict(pop_size=100, pc=.9, pm=1/5, max_gen=500)
crash_runs = [run_case(crash_problem, NSGA2, crash_config, seed) for seed in SEEDS]
crash_pool = np.vstack([feasible_front(r) for r in crash_runs])
crash_lo, crash_hi = crash_pool.min(axis=0), crash_pool.max(axis=0)
crash_stats = []
for r in crash_runs:
    F = feasible_front(r)
    spacing = normalized_spacing(F, crash_lo, crash_hi)
    crash_stats.append(dict(seed=r["seed"],
        hv=normalized_hypervolume_3d(F, crash_lo, crash_hi),
        nn_cv=spacing["cv_nn"], n_nd=len(F), seconds=r["seconds"]))
show_table(["Seed", "HV (higher)", "NN distance CV", "Nondominated count", "Seconds"],
           [[s[k] for k in ("seed", "hv", "nn_cv", "n_nd", "seconds")] for s in crash_stats])
optimum_x, optimum_fx = crash_runs[0]["X"], crash_runs[0]["F"]
fig = plt.figure(figsize=(11, 4.2))
ax = fig.add_subplot(121, projection="3d")
ax.scatter(optimum_fx[:, 0], optimum_fx[:, 1], optimum_fx[:, 2],
           c=optimum_fx[:, 0], cmap="viridis", s=19)
ax.set(xlabel="$f_1$: weight", ylabel="$f_2$: acceleration",
       zlabel="$f_3$: intrusion", title="Crashworthiness: seed 14")
ax.view_init(elev=23, azim=-52)
ax2 = fig.add_subplot(122)
norm = normalize_objectives(optimum_fx, crash_lo, crash_hi)
for row in norm:
    ax2.plot(range(3), row, alpha=.22, color=COLORS[0], lw=.8)
ax2.set(xticks=range(3), xticklabels=["Weight", "Acceleration", "Intrusion"],
        ylabel="Objective value on common normalized scale",
        title="Trade-offs across objectives")
finish_figure(fig, "crash_results")
display(Markdown("**Interpretation.** Standard NSGA-II crowding distance is a sum of "
    "one-dimensional neighbour gaps. On a curved three-objective surface, that "
    "score does not guarantee uniform surface coverage; boundary points receive "
    "infinite crowding distance, and finite populations and stochastic variation "
    "can leave uneven densities. Since crowding distance is normalized per "
    "objective, the raw numerical magnitude of weight is not by itself a valid "
    "explanation. Larger populations, better diversity preservation (for example "
    "reference-direction or nearest-neighbour selection), and repeated runs are "
    "plausible improvements. Alternative algorithms are suggestions, not tested "
    "claims here. The pooled scale and HV compare these runs only; no exact "
    "crashworthiness Pareto front is asserted."))


## Part 2. Constrained Multi-Objective Optimization

In this part, your task is to implement the parameter-less constraint handling technique in NSGA-II and use the constrained NSGA-II to solve constrained optimization problems.

### Introduction to a Constrained Optimization Problem: MW7

The MW test suite [3] is a constrained multi-objective optimization test suite proposed by Zhongwei Ma and Yong Wang. It is designed to evaluate the performance of constrained multi-objective optimization algorithms. MW7 is one of the test problems in the MW test suite. We have implemented the MW7() class and provided examples on how to use it below.

- Number of objectives: 2
- Number of decision variables: 15
- Number of constraints: 2

$$
\begin{aligned}
&\mathrm{minimize} \left\{
\begin{array}{l}
f_1(\mathbf{x})=g_3(\mathbf{x}) x_1 \\
f_2(\mathbf{x})=g_3(\mathbf{x}) \sqrt{1-\left(f_1(\mathbf{x}) / g_3(\mathbf{x})\right)^2}
\end{array}\:,\right. \\
&\mathrm{subject~to} \enspace c_1(\mathbf{x})=\left(1.2+0.4 \sin (4 l)^{16}\right)^2-(f_1(\mathbf{x}))^2-(f_2(\mathbf{x}))^2 \geq 0,\\
& \qquad \qquad c_2(\mathbf{x})=(f_1(\mathbf{x}))^2 + (f_2(\mathbf{x}))^2 - \left(1.15 - 0.2 \sin (4 l)^8\right)^2 \geq 0,\\
&\mathrm{where} \enspace g_3(\mathbf{x})=1+\sum_{i=2}^n 2\left(x_i+\left(x_{i-1}-0.5\right)^2-1\right)^2, \\
& \qquad \enspace l=\arctan \left(f_2(\mathbf{x}) / f_1(\mathbf{x})\right),\\
& \qquad \enspace 0 \leq x_i \leq 1, \enspace i=1,\ldots,n.
\end{aligned}
$$


[3] Ma, Z., & Wang, Y. (2019). Evolutionary constrained multiobjective optimization: Test suite construction and performance comparisons. IEEE Transactions on Evolutionary Computation, 23(6), 972-986.

In [ ]:
class MW7():

    def __init__(self):
        self.name = 'MW7'
        self.n_obj = 2  # number of objectives
        self.n_var = 15  # number of decision variables
        self.n_con = 2  # number of constraints

        self.lower = np.zeros(self.n_var)
        self.upper = np.ones(self.n_var)

    def evaluate(self, x):
        pop_size = len(x)
        f = np.zeros((pop_size, self.n_obj))
        c = np.zeros((pop_size, self.n_con))

        g3 = 1 + 2.0 * np.sum((x[:, self.n_obj - 1:] +
                               (x[:, self.n_obj - 2:-1] - 0.5)**2 - 1.0)**2,
                              axis=1)

        f[:, 0] = g3 * x[:, 0]
        f[:, 1] = g3 * (1 - (f[:, 0] / g3)**2)**0.5

        with np.errstate(divide='ignore'):
            l = np.arctan(f[:, 1] / f[:, 0])

        c[:, 0] = (1.2 + 0.4 * (np.sin(4 * l))**16)**2 - f[:, 0]**2 - f[:, 1]**2
        c[:, 1] = f[:, 0]**2 + f[:, 1]**2 - (1.15 - 0.2 * (np.sin(4 * l))**8)**2
        c = np.where(c < 0, -c, 0)  # the degree of constraint violation of x at each constraint

        return f, c

In [ ]:
# Create an instance of MW7
prob = MW7()
print(f"Problem = {prob.name}")
print(f"Number of objectives = {prob.n_obj}")
print(f"Number of variables = {prob.n_var}")
print(f"Number of constraints = {prob.n_con}")
print(f"Lower bounds = {prob.lower}")
print(f"Upper bounds = {prob.upper}")

In [ ]:
# Initialize a population with 1 solution and evaluate it
pop_size = 1
x = prob.lower + (prob.upper - prob.lower) * np.random.rand(
    pop_size, prob.n_var)
print("x = ", x)

f, c = prob.evaluate(x)
print(f"f(x) = {f}")
print(f"c(x) = {c}")
print(f"CV(x) = {np.sum(c, axis=1, keepdims=True)}")  # Constraint Violation

In [ ]:
# Initialize a population with 2 solution and evaluate them
pop_size = 2
x = prob.lower + (prob.upper - prob.lower) * np.random.rand(
    pop_size, prob.n_var)
print("x = ", x)

f, c = prob.evaluate(x)
print(f"f(x) = {f}")
print(f"c(x) = {c}")
print(f"CV(x) = {np.sum(c, axis=1, keepdims=True)}")

### Task: Implement Constrained NSGA-II and Solve MW7 (30 marks)
Your task is to **implement the parameter-less constraint handling technique in NSGA-II and use the constrained NSGA-II to solve the MW7 problem**.

You can refer to the example figures below for reference. The left figure (from [3]) displays the true Pareto front (represented by red points), the unconstrained Pareto front (represented by the blue line), and the feasible region (represented by the gray region). The right figure displays the final population obtained by NSGA-II. Your expected solutions should be similar to the gray circles in the right figure.


    
    


Note that sometimes the solutions obtained by NSGA-II may not cover all segments of the Pareto front (as shown below). **What are the possible reasons? Please share your thoughts in your report.**


    
    


*Course reference images are not imported. The figures below are generated from the executed code.*

In [ ]:
class CNSGA2(NSGA2):
    """NSGA-II with parameter-less constraint dominance.

    Feasible solutions dominate infeasible ones. Between infeasible solutions,
    lower total violation wins. Between feasible solutions, objective Pareto
    dominance applies. Equal positive violation does not break ties by F.
    """

    def _dominance_matrix(self, F, CV=None):
        if CV is None:
            return self._objective_dominance(F)
        CV = np.asarray(CV, dtype=float).reshape(-1)
        if CV.shape != (len(F),) or np.any(CV < 0) or not np.isfinite(CV).all():
            raise ValueError("CV must be a finite nonnegative vector")
        feasible = CV == 0.0
        objective_dominance = self._objective_dominance(F)
        both_feasible = feasible[:, None] & feasible[None, :]
        feasible_over_infeasible = feasible[:, None] & ~feasible[None, :]
        both_infeasible = ~feasible[:, None] & ~feasible[None, :]
        lower_violation = CV[:, None] < CV[None, :]
        return ((both_feasible & objective_dominance)
                | feasible_over_infeasible | (both_infeasible & lower_violation))


In [ ]:
# The course MW7 class returns violation magnitudes, NOT signed constraints.
probe_F = np.array([[100.,100.], [0.,0.], [1.,1.]])
probe_CV = np.array([0.,2.,1.])
ranks, _ = CNSGA2(pop_size=3).non_dominated_sort(probe_F, probe_CV)
assert ranks.tolist() == [0,2,1]  # feasibility dominates even better objectives
theta = .4
construct = np.zeros((1,15))
construct[0,0] = np.cos(theta)
for j in range(1,15):
    construct[0,j] = 1-(construct[0,j-1]-.5)**2
F,C = MW7().evaluate(construct)
assert np.all(C>=0) and np.isclose(np.linalg.norm(F[0]),1)
print('Constraint-dominance and MW7 interface checks passed.')


### Task 3 experiment: MW7

Baseline: N=100, G=500, pc=0.9, pm=1/15. Feasible solutions dominate infeasible solutions; among infeasible solutions lower summed violation wins; among feasible solutions Pareto dominance applies. No penalty coefficient is introduced. Strict zero violation is used, matching the supplied nonnegative violation interface.

The numerical reference is derived from the supplied equations. At angle theta, the smallest feasible radius is max(1, 1.15-0.2 sin(4 theta)^8). Sample 100,001 angles, filter globally dominated points, then retain 2,000 reference samples without interpolating across gaps. The radial lower boundary alone is not all Pareto-optimal. This numerical reference has three disconnected segments.


In [ ]:
mw_problem = MW7()
mw_config = dict(pop_size=100, pc=.9, pm=1/15, max_gen=500)
mw_reference = mw7_reference_front(n_points=2000)
mw_lo, mw_hi = mw_reference.min(axis=0), mw_reference.max(axis=0)
mw_runs = [run_case(mw_problem, CNSGA2, mw_config, seed) for seed in SEEDS]
mw_stats = []
for r in mw_runs:
    F = feasible_front(r)
    mw_stats.append(dict(seed=r["seed"], feasible=float(np.mean(r["CV"] == 0)),
        igd=normalized_igd(mw_reference, F, mw_lo, mw_hi),
        gd=normalized_gd(mw_reference, F, mw_lo, mw_hi),
        segments=segment_coverage(F, mw_reference, mw_lo, mw_hi),
        seconds=r["seconds"], max_cv=float(r["CV"].max())))
show_table(["Seed", "Feasible fraction", "IGD (lower)", "GD (lower)", "Segments hit / 3"],
           [[s[k] for k in ("seed", "feasible", "igd", "gd", "segments")] for s in mw_stats])
optimum_x, optimum_fx = mw_runs[0]["X"], mw_runs[0]["F"]
fig, axes = plt.subplots(1, 2, figsize=(11, 4.1))
draw_reference(axes[0], mw_reference, "Numerical constrained front")
theta = np.linspace(0, np.pi/2, 1000)
axes[0].plot(np.cos(theta), np.sin(theta), "--", c="#999999", lw=1,
             label="Unconstrained front")
axes[0].scatter(optimum_fx[:, 0], optimum_fx[:, 1], color=COLORS[0], s=17,
                label="Final population, seed 14")
axes[0].set(xlabel="$f_1$", ylabel="$f_2$", title="MW7: feasibility and coverage")
axes[0].legend(fontsize=8)
for i, r in enumerate(mw_runs):
    curve = [normalized_igd(mw_reference, h["F"][h["CV"] == 0], mw_lo, mw_hi)
             for h in r["history"]]
    axes[1].plot([h["gen"] for h in r["history"]], curve,
                 color=COLORS[i], label=str(r["seed"]))
axes[1].set(xlabel="Offspring generations", ylabel="Feasible-set normalized IGD",
            title="Seed sensitivity (missing feasible sets omitted)", yscale="log")
axes[1].legend(title="Seed", fontsize=8)
finish_figure(fig, "mw7_results")
mmean, msd = mean_sd([s["igd"] for s in mw_stats])
display(Markdown(f"**Observed result.** Mean feasible-set IGD = **{mmean:.5f} ± {msd:.5f}**; "
    f"segment-hit counts = {[s['segments'] for s in mw_stats]}. Disconnected front "
    "segments can be lost through random initialization, finite-population drift "
    "and selection pressure. Feasibility-first comparison can reject infeasible "
    "intermediate designs that would help move between feasible regions. SBX and "
    "polynomial mutation tend to explore near the current population, so more "
    "generations do not guarantee recovery after a segment is lost. Larger "
    "populations, repeated restarts or diversity-aware exploration may help. "
    "These mechanisms explain possible failures; they do not imply every seed "
    "must fail or that feasibility alone proves convergence."))


### Task: Solve the Assigned Real-World Constrained Multi-Objective Optimization Problem (30 Marks)

Your group has been assigned one **real-world constrained multi-objective optimization (RCM) problem**. Check the [Google Sheet](https://docs.google.com/spreadsheets/d/16YdUPUa9-odsxajJ2FukP3yenpzy3Wl2gUPET8h2TDo/edit?usp=sharing) to identify your group’s assigned problem.

The complete mathematical formulation, including the objectives, constraints, variable bounds, and references, is available in the following paper:

> A. Kumar et al., “A benchmark-suite of real-world constrained multi-objective optimization problems and some baseline results,” *Swarm and Evolutionary Computation*, vol. 67, 2021. DOI: [10.1016/j.swevo.2021.100961](https://doi.org/10.1016/j.swevo.2021.100961).

A local copy is also included in the project ZIP file at `related papers/RCM.pdf`.

#### Core Project Tasks

Complete the following tasks for your assigned RCM problem:

- **Implement the Problem Class:** Implement the objectives, constraints, and decision-variable bounds of the assigned problem as a Python class.
- **Apply Constrained NSGA-II:** Use your constrained NSGA-II implementation to solve the assigned problem.
- **Conduct Parameter Analysis:** Systematically evaluate different population sizes, crossover and mutation probabilities, and maximum numbers of generations.
- **Analyze Algorithm Performance:** Discuss how these parameter choices affect convergence, solution diversity, and overall optimization performance.
- **Report the Final Results:** Present the Pareto-optimal solutions obtained using your best-performing parameter configuration and provide a critical discussion of the results.

### Task 4 model: Group 14 / RCM13

RCM13 has seven variables, three minimized objectives and eleven signed inequalities g_j<=0. There are no equality constraints. The weight proxy and the two shaft-stress objectives conflict. Variable bounds and the tooth-count domain are enforced by repair; the constraint interface returns max(g_j,0).

| Variable | Conventional interpretation | Course bound/domain |
|---|---|---|
| x1 | Gear face width | [2.6,3.6] |
| x2 | Tooth module | [0.7,0.8] |
| x3 | Tooth count | {17,28}, literal printed set |
| x4,x5 | Shaft bearing spans | [7.3,8.3] |
| x6 | First shaft diameter | [2.9,3.9] |
| x7 | Second shaft diameter | [5.0,5.5] |

The source section does not state physical units, so none are invented. Main formula source: Kumar et al. (2021), supplied supplement p.9, section 1.1.13, equation (29). The code below makes every constant and source convention explicit. It does not import any external optimization library.


In [ ]:
"""RCM13 gear-box model; the default reproduces the supplied course PDF.

Only the engineering problem is defined here. No optimizer is imported.
"""

import numpy as np


class RCM13:
    """Seven-variable, three-objective gear-box design with 11 inequalities.

    Source: Kumar et al. (2021), DOI 10.1016/j.swevo.2021.100961,
    course ``RCM supp.pdf``, p. 9, section 1.1.13, equation (29).

    The conventional speed-reducer interpretation is x1 = gear face width,
    x2 = tooth module, x3 = pinion tooth count, x4/x5 = shaft bearing spans,
    and x6/x7 = shaft diameters. The supplied section does not state units;
    no physical unit or engineering certification is inferred here.
    f1 is the supplied weight/volume proxy; f2 and f3 are shaft stresses.
    All three objectives are minimized. All signed inequalities use g <= 0.

    ``literal_pdf`` (default) preserves the course equation exactly:
      f1 uses 14.9334/x3 and 3.3333*x3**2; x3 is the printed set {17, 28};
      f2 uses 16.91e6; stress limits are 1100 and 850.
    The set notation is ambiguous relative to publicly released code, so it
    is made an explicit modeling convention, not silently corrected.

    Two OPTIONAL source-specific sensitivity variants are kept separate:
      ``platemo``: the public RWMOP13.m formula uses 14.933*x3,
      (10/3)*x3**2, integer x3 in 17..28, 16.91e6, and 1100/850 limits.
      ``author_code``: CEC2021_func.m, func == 13, uses the same f1 as
      PlatEMO, integer x3 in 17..28, 16.9e6, and 1300/1100 limits.
    They are different problem definitions; results must never be pooled.

    Public cross-checks (accessed 2026-10-04):
    https://github.com/BIMK/PlatEMO/blob/master/PlatEMO/Problems/Multi-objective%20optimization/RWMOPs/RWMOP13.m
    https://github.com/P-N-Suganthan/2021-RW-MOP/blob/main/CEC2021-RWCMOP.zip

    ``repair`` clips bounds and applies the selected tooth-count convention.
    Call it when initializing and after variation, so stored designs equal
    evaluated designs. ``evaluate`` defensively repairs a copy as well and
    returns (F, C) with shapes (N, 3), (N, 11), C = maximum(g, 0).
    Constraint order follows the course PDF, including stress limits at
    columns 4 and 5 (zero-based). Bounds/discreteness are handled by repair.
    """

    name = "RCM13 Gear Box Design"
    n_obj = 3
    n_var = 7
    n_con = 11
    lower = np.array([2.6, 0.7, 17.0, 7.3, 7.3, 2.9, 5.0])
    upper = np.array([3.6, 0.8, 28.0, 8.3, 8.3, 3.9, 5.5])
    variable_names = ("face_width", "module", "teeth", "span_1", "span_2",
                      "diameter_1", "diameter_2")
    objective_names = ("weight_proxy", "shaft_1_stress", "shaft_2_stress")

    def __init__(self, formulation="literal_pdf"):
        allowed = ("literal_pdf", "platemo", "author_code")
        if formulation not in allowed:
            raise ValueError(f"formulation must be one of {allowed}")
        self.formulation = formulation
        self.lower = type(self).lower.copy()
        self.upper = type(self).upper.copy()
        self.stress_limits = np.array(
            [1300.0, 1100.0] if formulation == "author_code" else [1100.0, 850.0]
        )

    def repair(self, X):
        """Return a repaired copy, preserving vector-versus-matrix shape.

        Literal-PDF tooth count uses the nearest of {17, 28}, with ties
        assigned to 28. Reference variants round half-up to 17,...,28.
        """
        a = np.asarray(X, dtype=float)
        if a.ndim not in (1, 2) or a.shape[-1] != self.n_var:
            raise ValueError("X must have shape (7,) or (N, 7)")
        if not np.all(np.isfinite(a)):
            raise ValueError("X must contain finite decision values")
        y = np.clip(a, self.lower, self.upper)
        if self.formulation == "literal_pdf":
            y[..., 2] = np.where(y[..., 2] < 22.5, 17.0, 28.0)
        else:
            y[..., 2] = np.floor(y[..., 2] + 0.5)
        return y

    def _objectives_and_signed_constraints(self, X):
        x1, x2, x3, x4, x5, x6, x7 = np.atleast_2d(self.repair(X)).T
        if self.formulation == "literal_pdf":
            gear_term = 14.9334 / x3 - 43.0934 + 3.3333 * x3**2
        else:
            gear_term = (10.0 / 3.0) * x3**2 + 14.933 * x3 - 43.0934
        stress_constant = 16.9e6 if self.formulation == "author_code" else 16.91e6
        f1 = (0.7854 * x1 * x2**2 * gear_term
              + 0.7854 * (x5 * x7**2 + x4 * x6**2)
              - 1.508 * x1 * (x7**2 + x6**2)
              + 7.477 * (x7**3 + x6**3))
        f2 = 10.0 * np.sqrt(stress_constant + (745.0 * x4 / (x2 * x3))**2) / x6**3
        f3 = 10.0 * np.sqrt(157.5e6 + (745.0 * x5 / (x2 * x3))**2) / x7**3
        F = np.column_stack((f1, f2, f3))
        G = np.column_stack((
            1.0 / (x1 * x2**2 * x3) - 1.0 / 27.0,
            1.0 / (x1 * x2**2 * x3**2) - 1.0 / 397.5,
            x4**3 / (x2 * x6**4 * x3) - 1.0 / 1.93,
            x5**3 / (x2 * x7**4 * x3) - 1.0 / 1.93,
            f2 - self.stress_limits[0],
            f3 - self.stress_limits[1],
            x2 * x3 - 40.0,
            -x1 / x2 + 5.0,
            x1 / x2 - 12.0,
            1.5 * x6 - x4 + 1.9,
            1.1 * x7 - x5 + 1.9,
        ))
        return F, G

    def evaluate(self, X):
        """Evaluate repaired designs; return objectives and nonnegative CVs."""
        F, G = self._objectives_and_signed_constraints(X)
        return F, np.maximum(G, 0.0)

    def constraint_values(self, X):
        """Return signed PDF-order inequalities for independent feasibility QA."""
        return self._objectives_and_signed_constraints(X)[1]


In [ ]:
# Independent high-precision reference calculation, not an optimizer result.
known_x = np.array([[3.5,.7,17,7.8,8,3.6,5.4]])
known_f = np.array([[2807.2060504005,887.5748882928394,797.6356607479250]])
known_model = RCM13('literal_pdf')
observed_f, observed_c = known_model.evaluate(known_x)
np.testing.assert_allclose(observed_f,known_f,rtol=1e-11,atol=1e-9)
assert np.all(observed_c==0)
assert np.all(known_model.constraint_values(known_x)<=0)
to_repair = np.tile(known_x,(2,1)); to_repair[:,2] = [18,27]
assert known_model.repair(to_repair)[:,2].tolist()==[17.,28.]
print('RCM13 independent objective/constraint values and tooth-domain checks passed.')


### RCM13 parameter-analysis design

Nine unique one-factor-at-a-time (OFAT) configurations use five common seeds each. Baseline: N=100, pc=0.9, pm=1/7, G=500, eta_c=eta_m=20. Vary N in {50,100,200}, pc in {0.6,0.9,1.0}, pm in {0.5/7,1/7,2/7}, and G in {200,500,1000}. Three additional population controls use exactly 50,000 evaluations each; initialization counts toward that budget.

All 60 tuning/control runs use a single fixed pooled objective scale. Exact three-dimensional hypervolume (higher is better) uses a common normalized reference point (1.1,1.1,1.1). Empirical IGD (lower is better) uses a voxel-stratified nondominated union of those runs; this is not the true Pareto front. Feasibility, dispersion and evaluation counts are reported alongside quality.

Select the highest mean HV among the nine OFAT configurations, then compare that choice with baseline on five unseen holdout seeds. This checks generalization without claiming an exhaustive search or statistical significance. The final plot and full design matrix use the first predeclared holdout seed (1014), not the best-looking seed.


In [ ]:
rcm_problem = RCM13(formulation=RCM_FORMULATION)
rcm_baseline = dict(pop_size=100, pc=.9, pm=1/7, max_gen=500)
rcm_configs = {"baseline": dict(rcm_baseline)}
factor_specs = {
    "Population": ("pop_size", [50, 100, 200], ["N50", "baseline", "N200"]),
    "Crossover": ("pc", [.6, .9, 1.0], ["pc0.6", "baseline", "pc1.0"]),
    "Mutation": ("pm", [.5/7, 1/7, 2/7], ["pm0.5/n", "baseline", "pm2/n"]),
    "Generations": ("max_gen", [200, 500, 1000], ["G200", "baseline", "G1000"]),
}
for factor, (parameter, values, labels) in factor_specs.items():
    for value, label in zip(values, labels):
        rcm_configs[label] = dict(rcm_baseline, **{parameter: value})
tuning_labels = list(rcm_configs)
# Equal-evaluation control: each population receives exactly 50,000 evaluations,
# including initialization. This separates population size from extra compute.
for n in [50, 100, 200]:
    rcm_configs[f"budget_N{n}"] = dict(rcm_baseline, pop_size=n,
                                      max_gen=50000//n-1)
rcm_runs = {}
for label, config in rcm_configs.items():
    rcm_runs[label] = [run_case(rcm_problem, CNSGA2, config, seed) for seed in SEEDS]
    print(f"{label:12s}: {len(SEEDS)} seeds, N={config['pop_size']}, "
          f"G={config['max_gen']}, evaluations/run={rcm_runs[label][0]['n_eval']:,}")

# Freeze objective scaling ONCE for the complete tuning/control comparison.
# It is not a theoretical ideal/nadir; do not compare HV to other formulations.
rcm_pool = np.vstack([feasible_front(r) for runs in rcm_runs.values() for r in runs])
if not len(rcm_pool):
    raise RuntimeError("No RCM feasible solution found; do not report a false Pareto front")
rcm_lo, rcm_hi = rcm_pool.min(axis=0), rcm_pool.max(axis=0)
rcm_pooled_front = np.unique(rcm_pool[non_dominated(rcm_pool)], axis=0)
# A voxel-stratified empirical reference avoids simply weighting each dense run
# equally many times. It remains data-dependent and is not a known true front.
voxels = np.floor(normalize_objectives(rcm_pooled_front, rcm_lo, rcm_hi)*25).astype(int)
_, reference_indices = np.unique(voxels, axis=0, return_index=True)
rcm_reference = rcm_pooled_front[np.sort(reference_indices)]


def score_rcm_run(r):
    F = feasible_front(r)
    spacing = normalized_spacing(F, rcm_lo, rcm_hi)
    return dict(seed=r["seed"], feasible=float(np.mean(r["CV"] == 0)),
        max_cv=float(np.max(r["CV"])), n_nd=len(F),
        n_unique=len(np.unique(F, axis=0)),
        teeth_17=int(np.sum((r["X"][:, 2] == 17) & (r["CV"] == 0))),
        teeth_28=int(np.sum((r["X"][:, 2] == 28) & (r["CV"] == 0))),
        outside_hv_reference=int(np.sum(np.any(normalize_objectives(F, rcm_lo, rcm_hi) >= 1.1, axis=1))),
        hv=normalized_hypervolume_3d(F, rcm_lo, rcm_hi, reference=1.1),
        igd=normalized_igd(rcm_reference, F, rcm_lo, rcm_hi),
        nn_cv=spacing["cv_nn"], seconds=r["seconds"], n_eval=r["n_eval"])


rcm_scores = {label: [score_rcm_run(r) for r in runs] for label, runs in rcm_runs.items()}
rcm_summary = {}
for label, scores in rcm_scores.items():
    row = dict(config=rcm_configs[label])
    for key in ["hv", "igd", "feasible", "nn_cv", "seconds", "n_nd"]:
        row[key + "_mean"], row[key + "_sd"] = mean_sd([s[key] for s in scores])
    row["n_eval"] = scores[0]["n_eval"]
    rcm_summary[label] = row
show_table(["Config", "N", "pc", "pm", "G", "Evaluations", "HV mean", "HV SD", "Emp. IGD", "Feasible"],
    [[label, *[s["config"][k] for k in ["pop_size", "pc", "pm", "max_gen"]],
      s["n_eval"], s["hv_mean"], s["hv_sd"], s["igd_mean"], s["feasible_mean"]]
     for label, s in rcm_summary.items()])
best_label = max(tuning_labels, key=lambda label: rcm_summary[label]["hv_mean"])
best_config = rcm_configs[best_label]
display(Markdown(f"**Selected configuration:** `{best_label}` = `{best_config}`. "
    "Selection maximizes mean normalized HV over the five tuning seeds among "
    "the nine OFAT configurations. This is the best tested setting, not a global "
    "parameter optimum. The equal-budget controls are evaluated separately. "
    f"The empirical reference has {len(rcm_reference)} points; the normalization "
    f"bounds are `{rcm_lo.round(5).tolist()}` and `{rcm_hi.round(5).tolist()}`. "
    "All comparisons use the same bounds and HV reference (1.1, 1.1, 1.1)."))

fig, axes = plt.subplots(2, 2, figsize=(10.7, 7))
for ax, (factor, (parameter, values, labels)) in zip(axes.flat, factor_specs.items()):
    values_for_axis = [v*7 for v in values] if parameter == "pm" else values
    avg = [rcm_summary[label]["hv_mean"] for label in labels]
    sd = [rcm_summary[label]["hv_sd"] for label in labels]
    ax.errorbar(values_for_axis, avg, yerr=sd, marker="o", capsize=4,
                 color=COLORS[0], lw=1.5)
    ax.set(xlabel="Mutation multiplier / n_var" if parameter == "pm" else parameter,
           ylabel="Normalized HV (mean ± sample SD)", title=factor)
    ax.set_xticks(values_for_axis)
finish_figure(fig, "rcm_parameter_analysis")

# Holdout seeds were not used for parameter selection or normalization.
HOLDOUT_SEEDS = [1014, 1114, 1214, 1314, 1414]
holdout_runs = {"baseline": [run_case(rcm_problem, CNSGA2, rcm_baseline, seed)
                              for seed in HOLDOUT_SEEDS]}
holdout_runs["selected"] = ([run_case(rcm_problem, CNSGA2, best_config, seed)
                              for seed in HOLDOUT_SEEDS]
                            if best_label != "baseline" else holdout_runs["baseline"])
holdout_scores = {label: [score_rcm_run(r) for r in runs] for label, runs in holdout_runs.items()}
holdout_summary = {}
for label, scores in holdout_scores.items():
    holdout_summary[label] = {}
    for key in ["hv", "igd", "feasible", "nn_cv", "seconds"]:
        avg, sd = mean_sd([s[key] for s in scores])
        holdout_summary[label][key + "_mean"] = avg
        holdout_summary[label][key + "_sd"] = sd
show_table(["Holdout config", "HV mean", "HV SD", "Empirical IGD", "Feasible fraction"],
    [[label, s["hv_mean"], s["hv_sd"], s["igd_mean"], s["feasible_mean"]]
     for label, s in holdout_summary.items()])

# A predeclared holdout seed (1014), not the best-looking seed, supplies the
# representative final population and design table requested by the assignment.
rcm_final = holdout_runs["selected"][0]
optimum_x, optimum_fx = rcm_final["X"], rcm_final["F"]
rcm_feasible_mask = rcm_final["CV"] == 0.0
rcm_nd_mask = np.zeros(len(optimum_fx), dtype=bool)
rcm_nd_mask[rcm_feasible_mask] = non_dominated(optimum_fx[rcm_feasible_mask])
rcm_final_x, rcm_final_f = optimum_x[rcm_nd_mask], optimum_fx[rcm_nd_mask]
assert len(rcm_final_x) > 0
assert np.all(rcm_problem.constraint_values(rcm_final_x) <= 0)
assert np.array_equal(rcm_final_x, rcm_problem.repair(rcm_final_x))
norm = normalize_objectives(rcm_final_f, rcm_lo, rcm_hi)
display_indices = list(dict.fromkeys([*np.argmin(rcm_final_f, axis=0).tolist(),
                                     int(np.argmin(np.linalg.norm(norm, axis=1)))]))
show_table(["Design", *[f"x{i}" for i in range(1, 8)], "f1", "f2", "f3", "Max g"],
    [[i, *rcm_final_x[i], *rcm_final_f[i],
      float(rcm_problem.constraint_values(rcm_final_x[i]).max())] for i in display_indices])
print("Full final nondominated design matrix (rows align with objective matrix):")
with np.printoptions(precision=6, suppress=True, threshold=5000):
    print(rcm_final_x)
    print("Objective matrix:")
    print(rcm_final_f)

fig = plt.figure(figsize=(11, 4.4))
ax = fig.add_subplot(121, projection="3d")
for tooth, color in [(17, "#156082"), (28, "#E97132")]:
    selected_teeth = rcm_final_x[:, 2] == tooth
    ax.scatter(*rcm_final_f[selected_teeth].T, color=color, s=19, label=f"{tooth} teeth")
ax.set(xlabel="$f_1$: weight proxy", ylabel="$f_2$: shaft 1 stress",
       zlabel="$f_3$: shaft 2 stress", title=f"RCM13: {best_label}, holdout seed 1014")
ax.view_init(elev=22, azim=-53)
ax.legend(fontsize=8, loc="upper left")
ax2 = fig.add_subplot(122)
for label, color in [("baseline", COLORS[1]), ("selected", COLORS[0])]:
    run = holdout_runs[label][0]
    curve = [normalized_hypervolume_3d(h["F"][h["CV"] == 0], rcm_lo, rcm_hi)
             for h in run["history"]]
    ax2.plot([h["n_eval"] for h in run["history"]], curve, marker=".",
             color=color, label=label)
ax2.set(xlabel="Objective evaluations (including initialization)",
        ylabel="Normalized feasible-set HV", title="Convergence at a common scale")
ax2.legend()
finish_figure(fig, "rcm_final_results")

# Preserve the numerical experiment record outside the submission only when
# exporting is explicitly enabled by the build environment.
if ARTIFACT_DIR:
    np.savetxt(ARTIFACT_DIR/"rcm_final_designs.csv",
               np.column_stack((rcm_final_x, rcm_final_f,
                                rcm_problem.constraint_values(rcm_final_x))),
               delimiter=",", header=",".join([*[f"x{i}" for i in range(1,8)],
                    "f1","f2","f3", *[f"g{i}" for i in range(1,12)]]), comments="")
    rows = []
    for label, scores in rcm_scores.items():
        rows.extend(dict(label=label, **s, **rcm_configs[label]) for s in scores)
    with (ARTIFACT_DIR/"rcm_experiments.csv").open("w", newline="", encoding="utf-8") as stream:
        writer = csv.DictWriter(stream, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)


In [ ]:
def percent_change(new, old):
    return 100.0*(new/old-1.0) if old != 0 else float("nan")

bhv = rcm_summary["baseline"]["hv_mean"]
factor_lines = []
for factor, (_, values, labels) in factor_specs.items():
    best_factor_label = max(labels, key=lambda k: rcm_summary[k]["hv_mean"])
    factor_lines.append(f"- **{factor}:** mean HV values for {values} are "
        f"{[round(rcm_summary[k]['hv_mean'],5) for k in labels]}. "
        f"The strongest tested level in this slice is `{best_factor_label}`.")
gain = percent_change(holdout_summary["selected"]["hv_mean"],
                      holdout_summary["baseline"]["hv_mean"])
display(Markdown("### RCM13 findings and limitations\n\n" + "\n".join(factor_lines) +
    f"\n\nThe selected configuration changes holdout mean HV by **{gain:+.2f}%** "
    "relative to baseline. Five paired seeds describe stochastic variability; "
    "they are not a large-sample significance claim. OFAT isolates main effects "
    "around one baseline and does not estimate parameter interactions. More "
    "individuals or generations can improve coverage by spending more "
    "evaluations; the equal-budget population controls below help separate "
    "this from an intrinsic population-size benefit. Crossover exchanges "
    "existing building blocks; mutation helps explore new designs but excessive "
    "mutation can disrupt good solutions. Higher HV is assessed jointly with "
    "feasibility, empirical IGD and plots, not NN spacing alone.\n\n"
    "Larger shaft diameters reduce the stress objectives (which scale with "
    "inverse diameter cubed) while increasing the weight/volume terms. The "
    "resulting solutions expose engineering trade-offs rather than a unique "
    "best design. A minimum normalized-distance-to-ideal design in the table "
    "is an illustrative compromise, not a user-preference optimum.\n\n"
    "**Model-source limitation.** The main experiment uses the course PDF "
    "literally: 14.9334/x3 in f1, tooth count in {17,28}, and stress caps "
    "1100/850. Public implementations use different formulas/domains. These "
    "are different optimization problems; their results are never pooled into "
    "the main metrics. Clarify the intended convention with the instructor "
    "before final submission if a correction has been issued."))
show_table(["Equal budget", "N", "G", "Evaluations", "HV mean", "HV SD"],
    [[label, rcm_configs[label]["pop_size"], rcm_configs[label]["max_gen"],
      rcm_summary[label]["n_eval"], rcm_summary[label]["hv_mean"], rcm_summary[label]["hv_sd"]]
     for label in ["budget_N50", "budget_N100", "budget_N200"]])
display(Markdown("**Small-population diagnostic.** The table below checks whether "
    "a discrete tooth-count branch disappeared. With the literal two-point "
    "domain and local polynomial mutation, recreating a lost branch can be "
    "rare; this is a mechanism to inspect, not a universal explanation for "
    "every run. HV-reference exclusions are also reported for holdout runs."))
show_table(["N=50 seed", "HV", "17-tooth feasible designs", "28-tooth feasible designs"],
    [[s["seed"], s["hv"], s["teeth_17"], s["teeth_28"]] for s in rcm_scores["N50"]])
show_table(["Holdout config", "Seed", "Unique nondominated vectors", "Outside HV reference"],
    [[label, s["seed"], s["n_unique"], s["outside_hv_reference"]]
     for label, scores in holdout_scores.items() for s in scores])

source_sensitivity = []
probe = np.array([[3.5, .7, 17, 7.8, 8, 3.6, 5.4]])
for mode in ["literal_pdf", "platemo", "author_code"]:
    model = RCM13(mode)
    f_probe, c_probe = model.evaluate(probe)
    source_sensitivity.append(dict(mode=mode, F=f_probe[0].tolist(),
                                   CV=float(c_probe.sum())))
show_table(["Independent source definition", "Probe f1", "Probe f2", "Probe f3", "CV"],
    [[s["mode"], *s["F"], s["CV"]] for s in source_sensitivity])

summary = dict(group=GROUP, problem="RCM13", formulation=RCM_FORMULATION,
    seeds=SEEDS, holdout_seeds=HOLDOUT_SEEDS,
    versions=dict(python=sys.version.split()[0], numpy=np.__version__, matplotlib=matplotlib.__version__),
    zdt=zdt_stats, crash=crash_stats, mw7=mw_stats,
    rcm_configs=rcm_configs, rcm=rcm_summary, rcm_scores=rcm_scores,
    selected_label=best_label, selected_config=best_config,
    holdout=holdout_summary, holdout_scores=holdout_scores,
    normalization=dict(lo=rcm_lo, hi=rcm_hi, hv_reference=[1.1]*3,
                       empirical_reference_size=len(rcm_reference)),
    final=dict(seed=1014, n_feasible=int(np.sum(rcm_feasible_mask)),
               n_nondominated=len(rcm_final_f),
               n_unique=len(np.unique(rcm_final_f,axis=0)),
               max_signed_constraint=float(rcm_problem.constraint_values(rcm_final_x).max()),
               X=rcm_final_x, F=rcm_final_f, representative_indices=display_indices),
    source_sensitivity=source_sensitivity)
if ARTIFACT_DIR:
    (ARTIFACT_DIR/"summary.json").write_text(json.dumps(json_ready(summary), indent=2), encoding="utf-8")
print("All four task sections executed. Main RCM result uses:", RCM_FORMULATION)
print("Reference scale frozen; holdout seeds excluded from parameter selection.")


### Reproducibility and submission checklist

- This notebook has been executed from a fresh kernel; every result figure is produced by code.
- The three course problem classes remain identical to the supplied definitions.
- Algorithm checks cover objective/constraint dominance, reproducibility, bounds and an independent HV example; the RCM model has an independent numerical reference check.
- No `.py`, image, CSV or auxiliary file is needed to run this notebook. Optional exports are off unless the build environment enables them.
- Submit `CEG5302_Group_Project_14.ipynb` and the matching 2–3-page `report_14.pdf` inside `EC_Group_14.zip`, with no extra files or directories.
- Generated Pareto approximations are experimental results, not global-optimality certificates. Confirm any instructor correction to the RCM13 source convention before submission.

### References

1. Deb, K., Pratap, A., Agarwal, S., and Meyarivan, T. (2002). A fast and elitist multiobjective genetic algorithm: NSGA-II. *IEEE Transactions on Evolutionary Computation*, 6(2), 182–197. DOI: 10.1109/4235.996017.
2. Zitzler, E., Deb, K., and Thiele, L. (2000). Comparison of multiobjective evolutionary algorithms: Empirical results. *Evolutionary Computation*, 8(2), 173–195.
3. Liao, X. et al. (2008). Multiobjective optimization for crash safety design of vehicles using a stepwise regression model. *Structural and Multidisciplinary Optimization*, 35, 561–569.
4. Ma, Z. and Wang, Y. (2019). Evolutionary constrained multiobjective optimization: Test suite construction and performance comparisons. *IEEE Transactions on Evolutionary Computation*, 23(6), 972–986.
5. Kumar, A. et al. (2021). A benchmark-suite of real-world constrained multi-objective optimization problems and some baseline results. *Swarm and Evolutionary Computation*, 67, 100961. DOI: 10.1016/j.swevo.2021.100961. RCM13: supplied supplement p.9, equation (29).
6. Source discrepancy checks: [PlatEMO RWMOP13](https://github.com/BIMK/PlatEMO/blob/master/PlatEMO/Problems/Multi-objective%20optimization/RWMOPs/RWMOP13.m) and [authors' CEC2021-RWCMOP archive](https://github.com/P-N-Suganthan/2021-RW-MOP/blob/main/CEC2021-RWCMOP.zip), accessed 4 October 2026. These are alternative definitions, not the main run's problem source.
7. [Course assignment sheet](https://docs.google.com/spreadsheets/d/16YdUPUa9-odsxajJ2FukP3yenpzy3Wl2gUPET8h2TDo/edit), Group 14, RCM13, checked 4 October 2026.
